In [2]:
import sys
import os
from anthropic.types import ToolParam
from weather_tools import get_weather, get_weather_schema
from dotenv import load_dotenv
from anthropic import Anthropic
from anthropic.types import MessageParam
from typing import Any, cast

#Using same helper functions as the other notebooks for consistency and ease of use
sys.path.insert(0, os.path.join(os.path.dirname(__file__) if '__file__' in dir() else os.getcwd(), '..'))
from helper_functions import add_user_message, add_assistant_message, chat, DEFAULT_MODEL, client


# Tools and Schemas
from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass


In [3]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format or date_format.strip() == "":
        raise ValueError("date_format cannot be empty.")
    return datetime.now().strftime(date_format)
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the provided date_format string. The date_format parameter should follow Python's strftime format codes, allowing for flexible output formats. For example, using '%Y-%m-%d %H:%M:%S' will return the current date and time in the format '2026-03-29 14:30:00'.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "The format string for the output date and time, following Python's strftime format codes. Defaults to '%Y-%m-%d %H:%M:%S'.",
            }
        },
        "required": [],
    },
})


In [9]:
import json

# CALLING get curent datetime tool 

messages = []
messages.append({
    "role": "user",
    "content": "what is the exact time formatted as HH:MM:SS?"
})
client = Anthropic()
response = client.messages.create(
    model=DEFAULT_MODEL,
    tools=[get_current_datetime_schema],
    messages=messages,
    max_tokens=1000
)

# Tool call usage may change the way Claude sends back the response.
messages.append({
    "role": "assistant",
    "content": response.content,
})

print(json.dumps(messages, indent=2, ensure_ascii=False, default=str))

[
  {
    "role": "user",
    "content": "what is the exact time formatted as HH:MM:SS?"
  },
  {
    "role": "assistant",
    "content": [
      "ToolUseBlock(id='toolu_01LREfkRp1nrQegQ5x9HSHSW', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')"
    ]
  }
]


In [6]:
# # Generated by Claude Code.
# # Tool dispatcher — maps tool names to their Python functions
# TOOLS = {
#     "get_weather": lambda inp: get_weather(inp["location"]),
#     "add_duration_to_datetime": lambda inp: add_duration_to_datetime(**inp),
#     "set_reminder": lambda inp: set_reminder(inp["content"], inp["timestamp"]),
# }

# ALL_SCHEMAS = [
#     get_weather_schema,
#     add_duration_to_datetime_schema,
#     set_reminder_schema,
# ]

# def run_agent(user_message: str, tools=ALL_SCHEMAS, max_turns: int = 10) -> str:
#     """Agentic loop: call Claude, execute any tool calls, and loop until done."""
#     messages = [{"role": "user", "content": user_message}]

#     for _ in range(max_turns):
#         response = client.messages.create(
#             model=DEFAULT_MODEL,
#             max_tokens=1024,
#             tools=tools,
#             messages=messages,
#         )

#         # Append assistant response to history
#         messages.append({"role": "assistant", "content": response.content})

#         if response.stop_reason == "end_turn":
#             # Extract final text block
#             for block in response.content:
#                 if block.type == "text":
#                     return block.text
#             return ""

#         if response.stop_reason == "tool_use":
#             tool_results = []
#             for block in response.content:
#                 if block.type == "tool_use":
#                     print(f"[tool call] {block.name}({block.input})")
#                     try:
#                         result = TOOLS[block.name](block.input)
#                         result_content = json.dumps(result) if not isinstance(result, str) else result
#                         is_error = False
#                     except Exception as exc:
#                         result_content = str(exc)
#                         is_error = True
#                     print(f"[tool result] {result_content}\n")
#                     tool_results.append({
#                         "type": "tool_result",
#                         "tool_use_id": block.id,
#                         "content": result_content,
#                         "is_error": is_error,
#                     })
#             messages.append({"role": "user", "content": tool_results})

#     return "Max turns reached."


# # --- Try it out ---
# answer = run_agent("What's the weather like in Paris right now? Also, what date will it be 10 days from today (2026-03-29)?")
# print(answer)


[tool call] get_weather({'location': 'Paris'})
[tool result] name 'json' is not defined

[tool call] add_duration_to_datetime({'datetime_str': '2026-03-29', 'duration': 10, 'unit': 'days', 'input_format': '%Y-%m-%d'})
[tool result] Wednesday, April 08, 2026 12:00:00 AM

I was able to calculate the date for you: **10 days from March 29, 2026 will be Wednesday, April 8, 2026**.

However, I encountered an error when trying to get the weather information for Paris. There seems to be a technical issue with the weather service at the moment. You might want to check a weather website or app directly for the current conditions in Paris.
